In [ ]:
#pip install torch==2.4.1 torchaudio==2.4.1 torchvision==0.19.1 soundfile gdown

In [2]:
import os
import torch
import torch.nn as nn
import torchaudio
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [ ]:
import gdown
import zipfile

# ID del dataset
file_id = '1_5N85MmkbWL0iDABjBUq9D41eoXh2wOm'

# Construir la URL de descarga
url = f'https://drive.google.com/uc?id={file_id}'
extract_file = "Dataset.zip"

# Descargar el zip
gdown.download(url, extract_file, quiet=False)

print("ZIP descargado")

# Descomprimir
with zipfile.ZipFile(extract_file, 'r') as zip_ref:
    zip_ref.extractall(os.path.splitext(extract_file)[0])

print("ZIP descomprimido en:", os.path.splitext(extract_file)[0])

In [ ]:
# Modelo Unet 2D para Audio Super Resolution

class AttentionGate(nn.Module):
    """Módulo de atención para ponderar skip connections"""
    def __init__(self, F_g, F_l, F_int):

        super().__init__()

        # Basado en el paper "https://arxiv.org/abs/1804.03999"
        # https://github.com/ozan-oktay/Attention-Gated-Networks
        self.W_g = nn.Conv2d(F_g, F_int, kernel_size=1)
        self.W_x = nn.Conv2d(F_l, F_int, kernel_size=1)
        self.psi = nn.Conv2d(F_int, 1, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, g, x):
        # Interpolar g para que tenga el mismo tamaño que x
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)
        # Calcular atención
        att = self.sigmoid(self.psi(self.relu(self.W_g(g) + self.W_x(x))))

        return x * att


class DilatedBlock(nn.Module):
    """Bloque con capas convolucionales dilatadas para capturar contexto de largo alcance sin perder resolución."""
    def __init__(self, channels):

        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(3,1), dilation=(1,1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(6,2), dilation=(2,2)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(12,4), dilation=(4,4)),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x) + x


class ResBlock(nn.Module):
    """Bloque Residual para Attention ResUNet."""
    def __init__(self, in_ch, out_ch):
        
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=(7,3), padding=(3,1))
        self.norm1 = nn.GroupNorm(out_ch//4, out_ch) # GroupNorm para batchs pequeños
        self.relu1 = nn.LeakyReLU(0.2, inplace=True)
        
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=(7,3), padding=(3,1))
        self.norm2 = nn.GroupNorm(out_ch//4, out_ch)
        self.relu2 = nn.LeakyReLU(0.2, inplace=True)
        
        # Skip connection
        if in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.GroupNorm(out_ch//4, out_ch)
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.relu1(out)
        
        out = self.conv2(out)
        out = self.norm2(out)
        
        out += identity
        out = self.relu2(out)
        
        return out


class UNetAudio2D(nn.Module):
    """Arquitectura Attention Res-UNet 2D adaptada para audio super resolution."""
    def __init__(self):

        super().__init__()

        # Encoder
        # Entrada: (B, 2, F, T)
        self.enc1 = ResBlock(2, 32)
        self.pool1 = nn.MaxPool2d((2,2))

        self.enc2 = ResBlock(32, 64)
        self.pool2 = nn.MaxPool2d((2,2))

        self.enc3 = ResBlock(64, 128)
        self.pool3 = nn.MaxPool2d((2,2))

        self.enc4 = ResBlock(128, 256)
        self.pool4 = nn.MaxPool2d((2,2))

        # Bottleneck
        self.bottleneck_conv = ResBlock(256, 512)
        self.bottleneck_dilated = DilatedBlock(512)

        # Decoder
        self.up4 = self.up_block(512,256)
        self.dec4 = ResBlock(512,256)

        self.up3 = self.up_block(256,128)
        self.dec3 = ResBlock(256,128)

        self.up2 = self.up_block(128,64)
        self.dec2 = ResBlock(128,64)

        self.up1 = self.up_block(64,32)
        self.dec1 = ResBlock(64,32)

        # Attention gates
        self.att4 = AttentionGate(256,256,128)
        self.att3 = AttentionGate(128,128,64)
        self.att2 = AttentionGate(64,64,32)
        self.att1 = AttentionGate(32,32,16)

        # Output
        self.final = nn.Conv2d(32,2,kernel_size=1)

    def up_block(self, in_ch, out_ch):
        """Bloque de upsampling en frecuencia y tiempo"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),  # sub-pixel upsampling
            nn.LeakyReLU(0.2, inplace=True)
        )

    def match_size(self, x, ref):
        """Asegura que x tenga el mismo tamaño que ref"""
        if x.shape[-2:] != ref.shape[-2:]:
            x = F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)

        return x

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        # Bottleneck
        b = self.bottleneck_conv(self.pool4(e4))
        b = self.bottleneck_dilated(b)

        # Decoder con skip connections y attention gates
        up4 = self.match_size(self.up4(b), e4)
        d4 = self.dec4(torch.cat([up4, self.att4(up4, e4)], dim=1))

        up3 = self.match_size(self.up3(d4), e3)
        d3 = self.dec3(torch.cat([up3, self.att3(up3, e3)], dim=1))

        up2 = self.match_size(self.up2(d3), e2)
        d2 = self.dec2(torch.cat([up2, self.att2(up2, e2)], dim=1))

        up1 = self.match_size(self.up1(d2), e1)
        d1 = self.dec1(torch.cat([up1, self.att1(up1, e1)], dim=1))

        return self.final(d1) + x

In [ ]:
# Discriminador Multi-Escala (MSD) y Multi-Periodo (MPD)
# Basado en HiFi-GAN
# https://arxiv.org/pdf/2010.05646

from torch.nn.utils import weight_norm, spectral_norm

def get_padding(kernel_size, dilation=1):
    return int((kernel_size*dilation - dilation)/2)

class DiscriminatorP(torch.nn.Module):
    def __init__(self, period, kernel_size=5, stride=3, use_spectral_norm=False):
        super(DiscriminatorP, self).__init__()
        self.period = period
        norm_f = weight_norm if use_spectral_norm == False else spectral_norm
        self.convs = nn.ModuleList([
            norm_f(nn.Conv2d(1, 32, (kernel_size, 1), (stride, 1), padding=(get_padding(5, 1), 0))),
            norm_f(nn.Conv2d(32, 128, (kernel_size, 1), (stride, 1), padding=(get_padding(5, 1), 0))),
            norm_f(nn.Conv2d(128, 512, (kernel_size, 1), (stride, 1), padding=(get_padding(5, 1), 0))),
            norm_f(nn.Conv2d(512, 1024, (kernel_size, 1), (stride, 1), padding=(get_padding(5, 1), 0))),
            norm_f(nn.Conv2d(1024, 1024, (kernel_size, 1), 1, padding=(2, 0))),
        ])
        self.conv_post = norm_f(nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0)))

    def forward(self, x):
        fmap = []

        # 1d to 2d
        b, c, t = x.shape
        if t % self.period != 0: # pad first
            n_pad = self.period - (t % self.period)
            x = F.pad(x, (0, n_pad), "reflect")
            t = t + n_pad
        x = x.view(b, c, t // self.period, self.period)

        for l in self.convs:
            x = l(x)
            x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)

        return x, fmap


class MultiPeriodDiscriminator(torch.nn.Module):
    def __init__(self):
        super(MultiPeriodDiscriminator, self).__init__()
        self.discriminators = nn.ModuleList([
            DiscriminatorP(2),
            DiscriminatorP(3),
            DiscriminatorP(5),
            DiscriminatorP(7),
            DiscriminatorP(11),
        ])

    def forward(self, y, y_hat):
        y_d_rs = []
        y_d_gs = []
        fmap_rs = []
        fmap_gs = []
        for i, d in enumerate(self.discriminators):
            y_d_r, fmap_r = d(y)
            y_d_g, fmap_g = d(y_hat)
            y_d_rs.append(y_d_r)
            fmap_rs.append(fmap_r)
            y_d_gs.append(y_d_g)
            fmap_gs.append(fmap_g)

        return y_d_rs, y_d_gs, fmap_rs, fmap_gs


class DiscriminatorS(torch.nn.Module):
    def __init__(self, use_spectral_norm=False):
        super(DiscriminatorS, self).__init__()
        norm_f = weight_norm if use_spectral_norm == False else spectral_norm
        self.convs = nn.ModuleList([
            norm_f(nn.Conv1d(1, 128, 15, 1, padding=7)),
            norm_f(nn.Conv1d(128, 128, 41, 2, groups=4, padding=20)),
            norm_f(nn.Conv1d(128, 256, 41, 2, groups=16, padding=20)),
            norm_f(nn.Conv1d(256, 512, 41, 4, groups=16, padding=20)),
            norm_f(nn.Conv1d(512, 1024, 41, 4, groups=16, padding=20)),
            norm_f(nn.Conv1d(1024, 1024, 41, 1, groups=16, padding=20)),
            norm_f(nn.Conv1d(1024, 1024, 5, 1, padding=2)),
        ])
        self.conv_post = norm_f(nn.Conv1d(1024, 1, 3, 1, padding=1))

    def forward(self, x):
        fmap = []
        for l in self.convs:
            x = l(x)
            x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)

        return x, fmap


class MultiScaleDiscriminator(torch.nn.Module):
    def __init__(self):
        super(MultiScaleDiscriminator, self).__init__()
        self.discriminators = nn.ModuleList([
            DiscriminatorS(use_spectral_norm=True),
            DiscriminatorS(),
        ])
        self.meanpools = nn.ModuleList([
            nn.AvgPool1d(4, 2, padding=2),
        ])

    def forward(self, y, y_hat):
        y_d_rs = []
        y_d_gs = []
        fmap_rs = []
        fmap_gs = []
        for i, d in enumerate(self.discriminators):
            if i != 0:
                y = self.meanpools[i-1](y)
                y_hat = self.meanpools[i-1](y_hat)
            y_d_r, fmap_r = d(y)
            y_d_g, fmap_g = d(y_hat)
            y_d_rs.append(y_d_r)
            fmap_rs.append(fmap_r)
            y_d_gs.append(y_d_g)
            fmap_gs.append(fmap_g)

        return y_d_rs, y_d_gs, fmap_rs, fmap_gs

class CombinedDiscriminator(nn.Module):
    """
    Discriminador que combina MSD y MPD
    """
    def __init__(self):

        super().__init__()
        
        self.msd = MultiScaleDiscriminator()
        self.mpd = MultiPeriodDiscriminator()

    def forward(self, y, y_hat):
        y_d_rs, y_d_gs, fmap_rs, fmap_gs = [], [], [], []
        
        for d in [self.msd, self.mpd]:
            y_r, y_g, fmap_r, fmap_g = d(y, y_hat)
            y_d_rs.extend(y_r)
            y_d_gs.extend(y_g)
            fmap_rs.extend(fmap_r)
            fmap_gs.extend(fmap_g)
            
        return y_d_rs, y_d_gs, fmap_rs, fmap_gs

In [ ]:
# Script para cargar el dataset
# Devuelve pares de STFT (real + imaginario) con shape (2, F, T) en fragmentos de 1.5s

NFFT = 1024
HOP_LENGTH = 256
SAMPLE_RATE = 44100
FRAGMENT_LENGTH = 65536

class AudioSuperResDataset(Dataset):
    """
    Carga pares de audio en Alta Calidad (HR) y Baja Calidad (LR),
    divide en fragmentos de FRAGMENT_LENGTH muestras y devuelve
    sus representaciones STFT con shape (2, F, T).

    hr_dir: Directorio de los archivos en alta calidad (Ground Truth)
    lr_dir: Directorio de los archivos en baja calidad (Input)
    fragment_length: Longitud del fragmento a procesar en muestras
    n_fft: Tamaño de la FFT para el STFT.
    hop_length: Salto entre ventanas STFT.
    """
    def __init__(self, hr_dir, lr_dir, fragment_length=FRAGMENT_LENGTH, n_fft=NFFT, hop_length=HOP_LENGTH):

        self.hr_dir = hr_dir
        self.lr_dir = lr_dir
        self.fragment_length = fragment_length
        self.n_fft = n_fft
        self.hop_length = hop_length

        # POOL_FACTOR = 2^4 = 16 (4 capas de pooling en la UNet 2D)
        self.pool_factor = 16

        # Cache de resamplers
        self._resamplers = {}

        # Ventana para STFT
        self.window = torch.hann_window(self.n_fft)

        # Contar ficheros
        files = [
            f for f in os.listdir(hr_dir)
            if f.endswith('.wav') and os.path.exists(os.path.join(lr_dir, f))
        ]

        # Cargar ficheros y dividir en fragmentos
        self.fragments = []  # lista (hr_fragment, lr_fragment)
        for filename in files:
            hr_path = os.path.join(hr_dir, filename)
            lr_path = os.path.join(lr_dir, filename)

            # Cargar los audios
            waveform_hr, sr_hr = torchaudio.load(hr_path)
            waveform_lr, sr_lr = torchaudio.load(lr_path)

            # Uniformizar a mono
            if waveform_hr.size(0) > 1:
                waveform_hr = waveform_hr.mean(dim=0, keepdim=True)
            if waveform_lr.size(0) > 1:
                waveform_lr = waveform_lr.mean(dim=0, keepdim=True)

            # Resamplear a 44.1 kHz
            if sr_hr != SAMPLE_RATE:
                key = ('hr', sr_hr)
                if key not in self._resamplers:
                    self._resamplers[key] = torchaudio.transforms.Resample(sr_hr, SAMPLE_RATE)
                waveform_hr = self._resamplers[key](waveform_hr)

            if sr_lr != SAMPLE_RATE:
                key = ('lr', sr_lr)
                if key not in self._resamplers:
                    self._resamplers[key] = torchaudio.transforms.Resample(sr_lr, SAMPLE_RATE)
                waveform_lr = self._resamplers[key](waveform_lr)

            # Normalizar a [-1, 1]
            max_val = max(waveform_hr.abs().max(), waveform_lr.abs().max()) + 1e-8
            waveform_hr = waveform_hr / max_val
            waveform_lr = waveform_lr / max_val

            # Dividir en fragmentos de fragment_length muestras
            min_len = min(waveform_hr.size(1), waveform_lr.size(1))
            num_fragments = min_len // self.fragment_length

            for i in range(num_fragments):
                start = i * self.fragment_length
                end = start + self.fragment_length
                frag_hr = waveform_hr[:, start:end]
                frag_lr = waveform_lr[:, start:end]

                # Descartar fragmentos de silencio
                if frag_hr.abs().max() < 1e-4:
                    continue

                self.fragments.append((frag_hr, frag_lr))

    def __len__(self):
        return len(self.fragments)

    def _waveform_to_stft(self, waveform):
        """
        Convierte una waveform (1, L) a un tensor STFT de shape (2, F, T)
        donde canal 0 = parte real y canal 1 = parte imaginaria.
        """
        stft = torch.stft(
            waveform.squeeze(0),          #(L,)
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.n_fft,
            window=self.window,
            return_complex=True,
        )  #stft shape: (F, T)

        # Separar parte real e imaginaria
        stft_ri = torch.stack([stft.real, stft.imag], dim=0)
        return stft_ri

    def _normalize_stft(self, stft_ri):
        """Compresión logarítmica de la magnitud, preservando la fase."""
        real = stft_ri[0]
        imag = stft_ri[1]
        
        magnitude = torch.sqrt(real**2 + imag**2 + 1e-8)
        phase_cos = real / magnitude  # cos(phase)
        phase_sin = imag / magnitude  # sin(phase)
        
        mag_compressed = torch.log1p(magnitude)
        
        return torch.stack([mag_compressed * phase_cos, mag_compressed * phase_sin], dim=0)

    def _pad_stft_to_pool_factor(self, stft):
        """
        Asegura que tanto F como T sean divisibles por pool_factor.
        Padding con ceros si es necesario.
        """
        _, freq_bins, time_frames = stft.shape

        # Pad frecuencia
        pad_f = (self.pool_factor - (freq_bins % self.pool_factor)) % self.pool_factor
        # Pad tiempo
        pad_t = (self.pool_factor - (time_frames % self.pool_factor)) % self.pool_factor

        if pad_f > 0 or pad_t > 0:
            stft = F.pad(stft, (0, pad_t, 0, pad_f))

        return stft

    def __getitem__(self, idx):
        frag_hr, frag_lr = self.fragments[idx]

        # Calcular STFT de ambas waveforms
        stft_lr = self._waveform_to_stft(frag_lr)
        stft_hr = self._waveform_to_stft(frag_hr)

        # Normalizar STFT con log-compresión para reducir rango dinámico
        stft_lr = self._normalize_stft(stft_lr)
        stft_hr = self._normalize_stft(stft_hr)

        # Asegurar que F y T son divisibles por pool_factor (16)
        stft_lr = self._pad_stft_to_pool_factor(stft_lr)
        stft_hr = self._pad_stft_to_pool_factor(stft_hr)

        # Devolver STFT: Input LR y Target HR, ambos con shape (2, F, T)
        return stft_lr, stft_hr

In [ ]:
# Loss para Audio Super-Resolución y el Discriminador
# Este script contiene las funciones de pérdida para entrenar la red neuronal, calculando
# la pérdida en el dominio del tiempo y en el dominio de la frecuencia.
# Basado en la Loss implementada en AERO: https://github.com/slp-rl/aero

NFFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
SAMPLE_RATE = 44100
FRAGMENT_LENGTH = 65536

class STFTLoss(nn.Module):
    """Pérdida STFT."""
    def __init__(self, n_fft=NFFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH):

        super().__init__()

        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.register_buffer("window", torch.hann_window(win_length))

    def forward(self, x, y):
        # x, y shape: (B, T)
        x_stft = torch.stft(x, self.n_fft, self.hop_length, self.win_length, self.window, return_complex=True)
        y_stft = torch.stft(y, self.n_fft, self.hop_length, self.win_length, self.window, return_complex=True)

        x_mag = torch.abs(x_stft)
        y_mag = torch.abs(y_stft)

        # Spectral Convergence Loss
        sc_loss = torch.norm(y_mag - x_mag, p="fro") / (torch.norm(y_mag, p="fro") + 1e-7)
        
        # Log-Magnitude Loss
        mag_loss = F.l1_loss(torch.log(x_mag + 1e-7), torch.log(y_mag + 1e-7))

        return sc_loss, mag_loss


class MultiResolutionSTFTLoss(nn.Module):
    """
    Pérdida STFT multi-resolución.
    Resoluciones basadas en AERO
    """
    def __init__(self, fft_sizes=[1024, 2048, 512], hop_sizes=[120, 240, 50], win_lengths=[600, 1200, 240]):


        super().__init__()


        assert len(fft_sizes) == len(hop_sizes) == len(win_lengths)
        
        # STFT diferentes resoluciones
        self.stft_losses = nn.ModuleList()
        for fs, ss, wl in zip(fft_sizes, hop_sizes, win_lengths):
            self.stft_losses.append(STFTLoss(fs, ss, wl))

    def forward(self, x, y):
        sc_loss = 0.0
        mag_loss = 0.0
        
        for f in self.stft_losses:
            sc_l, mag_l = f(x, y)
            sc_loss += sc_l
            mag_loss += mag_l
        
        sc_loss /= len(self.stft_losses)
        mag_loss /= len(self.stft_losses)
        
        return sc_loss + mag_loss


class CombinedLoss(nn.Module):
    """Pérdida combinada de L1 y MR-STFT."""

    def __init__(self, n_fft=NFFT, hop_length=HOP_LENGTH, lambda_l1=1.0, lambda_mrstft=1.0):

        super().__init__()

        self.n_fft = n_fft
        self.hop_length = hop_length
        
        self.lambda_l1 = lambda_l1
        self.lambda_mrstft = lambda_mrstft
        
        # Buffer ISTFT
        self.register_buffer('base_window', torch.hann_window(n_fft))
        
        # MRSTFT
        self.mrstft_loss = MultiResolutionSTFTLoss()

    def _denormalize_stft(self, stft):
        """Inversa de log-compresión."""
        real = stft[:, 0]
        imag = stft[:, 1]
        
        mag_compressed = torch.sqrt(real**2 + imag**2 + 1e-8)
        phase_cos = real / mag_compressed
        phase_sin = imag / mag_compressed
        
        magnitude = torch.exp(mag_compressed) - 1
        
        return torch.stack([magnitude * phase_cos, magnitude * phase_sin], dim=1)

    def _stft_to_waveform(self, stft_ri):
        """Aplica ISTFT."""
        # Deshacer padding en frecuencia
        valid_freq_bins = self.n_fft // 2 + 1
        if stft_ri.size(2) > valid_freq_bins:
            stft_ri = stft_ri[:, :, :valid_freq_bins, :]
            
        # Deshacer padding en tiempo
        valid_time_frames = FRAGMENT_LENGTH // self.hop_length + 1
        if stft_ri.size(3) > valid_time_frames:
            stft_ri = stft_ri[:, :, :, :valid_time_frames]

        stft_complex = torch.complex(stft_ri[:, 0], stft_ri[:, 1])
        waveform = torch.istft(
            stft_complex,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.n_fft,
            window=self.base_window,
            length=FRAGMENT_LENGTH
        )
        return waveform

    def get_waveforms(self, pred_stft, target_stft):
        """Pasar de STFT log-comprimido a waveform"""
        pred_linear = self._denormalize_stft(pred_stft)
        target_linear = self._denormalize_stft(target_stft)
        
        pred_wav = self._stft_to_waveform(pred_linear)
        target_wav = self._stft_to_waveform(target_linear)
    
        # Recortar a la misma longitud mínima
        min_len = min(pred_wav.size(-1), target_wav.size(-1))
        pred_wav = pred_wav[..., :min_len]
        target_wav = target_wav[..., :min_len]
    
        # Asegurar shape (B, 1, T) para el discriminador 1D
        if pred_wav.ndim == 2:
            pred_wav = pred_wav.unsqueeze(1)
            target_wav = target_wav.unsqueeze(1)
        
        return pred_wav, target_wav

    def forward(self, pred_stft, target_stft, pred_wav=None, target_wav=None):
        # Si no se proporcionan waveforms precalculadas, calcular internamente
        if pred_wav is None or target_wav is None:
            # Descomprimir
            pred_linear = self._denormalize_stft(pred_stft)
            target_linear = self._denormalize_stft(target_stft)
            
            # Reconstrucción a Waveform
            pred_wav = self._stft_to_waveform(pred_linear)
            target_wav = self._stft_to_waveform(target_linear)
        
        # Recortar a longitud mínima
        min_len = min(pred_wav.size(-1), target_wav.size(-1))
        pred_wav = pred_wav[..., :min_len]
        target_wav = target_wav[..., :min_len]
        
        # L1 loss
        loss_l1 = F.l1_loss(pred_wav, target_wav)
        
        # MRSTFT loss
        if pred_wav.ndim == 3:
            pred_wav = pred_wav.squeeze(1)
            target_wav = target_wav.squeeze(1)
        loss_mrstft = self.mrstft_loss(pred_wav, target_wav)
        
        return self.lambda_l1 * loss_l1 + self.lambda_mrstft * loss_mrstft


# Pérdidas del discriminador
class DiscriminatorLoss(nn.Module):
    """Pérdidas del discriminador."""
    @staticmethod
    def discriminator_loss(disc_real_outputs, disc_generated_outputs):
        """Pérdida LSGAN para el discriminador."""
        loss = 0
        for dr, dg in zip(disc_real_outputs, disc_generated_outputs):
            dr_loss = torch.mean((1 - dr)**2)
            dg_loss = torch.mean(dg**2)
            loss += (dr_loss + dg_loss)
        return loss

    @staticmethod
    def generator_loss(disc_generated_outputs):
        """Pérdida adversarial para el generador."""
        loss = 0
        for dg in disc_generated_outputs:
            loss += torch.mean((1 - dg)**2)
        return loss

    @staticmethod
    def feature_matching_loss(fmap_r, fmap_g):
        """Pérdida L1 entre los feature maps del discriminador."""
        loss = 0
        for dr, dg in zip(fmap_r, fmap_g):
            for rl, gl in zip(dr, dg):
                loss += torch.mean(torch.abs(rl - gl))
        return loss * 2 # 2x FM loss

In [ ]:
# Script para entrenar la red neuronal
# Este script guardara el mejor modelo en un archivo .pt

TRAIN_HR_DIR = './Dataset/train/HR'    # Archivos de alta resolución (output de la red)
TRAIN_LR_DIR = './Dataset/train/LR'    # Archivos de baja resolución (input de la red)
VAL_HR_DIR = './Dataset/test/HR'       # Archivos de alta resolución para validación
VAL_LR_DIR = './Dataset/test/LR'       # Archivos de baja resolución para validación

BATCH_SIZE = 8                      # Tamaño de lote
EPOCHS = 500                        # Épocas
LEARNING_RATE_G = 1e-4              # LR del generador
LEARNING_RATE_D = 1e-4              # LR del discriminador

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def evaluate(model_g, dataloader, criterion):
    """Evalúa el modelo en el conjunto de validación y devuelve la pérdida promedio."""
    model_g.eval()
    total_loss = 0.0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model_g(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
    return total_loss / len(dataloader)

def train():
    """Entrenar el modelo"""
    dirs_to_check = [TRAIN_HR_DIR, TRAIN_LR_DIR, VAL_HR_DIR, VAL_LR_DIR]
    for d in dirs_to_check:
        if not os.path.exists(d):
            print(f"Error: directorio no encontrado: {d}")
            return

    # Cargar datasets
    train_dataset = AudioSuperResDataset(TRAIN_HR_DIR, TRAIN_LR_DIR)
    val_dataset = AudioSuperResDataset(VAL_HR_DIR, VAL_LR_DIR)

    if len(train_dataset) == 0 or len(val_dataset) == 0:
        print("Error: El dataset está vacío")
        return

    num_workers = max(0, os.cpu_count()-1)
    print(f"Cargados {len(train_dataset)} elementos de entrenamiento con shapes de entrada y salida: {train_dataset[0][0].shape} y {train_dataset[0][1].shape}")
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True)
    print(f"Cargados {len(val_dataset)} elementos de validación con shapes de entrada y salida: {val_dataset[0][0].shape} y {val_dataset[0][1].shape}")
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)

    # Inicializar Modelos
    model_g = UNetAudio2D().to(DEVICE)
    model_d = CombinedDiscriminator().to(DEVICE)

    # Inicializar Pérdidas
    criterion_g = CombinedLoss(lambda_l1=0.5, lambda_mrstft=0.5).to(DEVICE)

    # Inicializar Optimizadores
    optimizer_g = optim.AdamW(model_g.parameters(), lr=LEARNING_RATE_G, betas=(0.8, 0.99))
    optimizer_d = optim.AdamW(model_d.parameters(), lr=LEARNING_RATE_D, betas=(0.8, 0.99))

    # Schedulers
    scheduler_g = optim.lr_scheduler.ReduceLROnPlateau(optimizer_g, mode='min', factor=0.5, patience=10)
    scheduler_d = optim.lr_scheduler.ReduceLROnPlateau(optimizer_d, mode='min', factor=0.5, patience=10)

    print(f"Iniciando entrenamiento en {DEVICE}...")

    best_val_loss = float('inf')
    patience_earlystop = 50
    warmup_epochs = 5
    epochs_no_improve = 0

    # Bucle de entrenamiento
    for epoch in range(EPOCHS):
        # Entrenar modelos
        model_g.train()
        model_d.train()

        running_loss_g = 0.0
        running_loss_d = 0.0

        for i, (inputs, targets) in enumerate(train_dataloader):
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)
            
            # Entrenar generador
            pred = model_g(inputs)

            # Obtener waveforms de LR y HR
            pred_wav, target_wav = criterion_g.get_waveforms(pred, targets)
            
            # Solo considerar el discriminador después del warmup
            if epoch >= warmup_epochs:
                optimizer_d.zero_grad(set_to_none=True)
                # Entrenar discriminador
                y_d_rs, y_d_gs, _, _ = model_d(target_wav.detach(), pred_wav.detach())
                # Calcular pérdidas
                loss_d = DiscriminatorLoss.discriminator_loss(y_d_rs, y_d_gs)
                loss_d.backward()
                # Gradient clipping para evitar explosión de gradientes
                torch.nn.utils.clip_grad_norm_(model_d.parameters(), max_norm=1.0)
                # Actualizar los pesos del modelo
                optimizer_d.step()
                running_loss_d += loss_d.item()
            
            optimizer_g.zero_grad(set_to_none=True)
            
            if epoch >= warmup_epochs:
                # Calcular pérdidas
                loss_g = criterion_g(pred, targets, pred_wav=pred_wav, target_wav=target_wav.detach())
                y_d_rs, y_d_gs, fmap_rs, fmap_gs = model_d(target_wav.detach(), pred_wav)
                loss_adv = DiscriminatorLoss.generator_loss(y_d_gs)
                loss_fm  = DiscriminatorLoss.feature_matching_loss(fmap_rs, fmap_gs)
                loss_g = loss_g + loss_adv + loss_fm
            else:
                loss_g = criterion_g(pred, targets)
            
            # Backward
            loss_g.backward()
            
            # Gradient clipping para evitar explosión de gradientes
            torch.nn.utils.clip_grad_norm_(model_g.parameters(), max_norm=1.0)
            
            # Actualizar los pesos del modelo
            optimizer_g.step()

            running_loss_g += loss_g.item()
        
        # Métricas
        train_loss_g = running_loss_g / len(train_dataloader)
        train_loss_d = running_loss_d / len(train_dataloader)

        # Evaluar generador en el conjunto de validación
        val_loss = evaluate(model_g, val_dataloader, criterion_g)

        print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss G: {train_loss_g:.6f} | Loss D: {train_loss_d:.6f} | Val Loss: {val_loss:.6f} | LR: {optimizer_g.param_groups[0]['lr']:.6f}")

        # Schedulers basados en val_loss
        scheduler_g.step(val_loss)
        scheduler_d.step(val_loss)

        # Guardar mejor modelo según val_loss
        if epoch >= warmup_epochs:  # Solo considerar guardar el modelo después de algunas épocas para evitar guardar modelos muy malos al inicio
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                torch.save(model_g.state_dict(), 'unet2D_superres_best.pt')
                print(f"Mejor modelo guardado con loss: {best_val_loss:.6f}")
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience_earlystop:
                    print(f"Early stopping en epoch {epoch+1}")
                    break
        
        torch.save(model_g.state_dict(), 'unet2D_superres.pt')

    print("Entrenamiento completado")
    print(f"Mejor loss alcanzado: {best_val_loss:.6f}")

In [ ]:
train()